# 04 — Statistical Analysis

Validates business hypotheses with formal statistical tests at α = 0.05.
Every test's statistic, p-value, effect size and plain-English conclusion
is written to `data/processed/statistical_test_results.csv`.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from pathlib import Path

ALPHA = 0.05
PROCESSED_DIR = Path("../data/processed")

trips = pd.read_csv(PROCESSED_DIR / "trips_analysis_ready.csv", parse_dates=["request_datetime"])
completed = trips.loc[trips.trip_status == "Completed"].copy()

results = []

def record(test_name, question, statistic, p_value, effect_size, effect_label, conclusion):
    results.append({
        "test_name": test_name,
        "business_question": question,
        "statistic": round(float(statistic), 4),
        "p_value": round(float(p_value), 6),
        "significant_at_0.05": bool(p_value < ALPHA),
        "effect_size": round(float(effect_size), 4) if effect_size is not None else None,
        "effect_size_metric": effect_label,
        "conclusion": conclusion,
    })

## 1. Chi-Square Test — Surge pricing vs. cancellation

H0: Surge-pricing tier is independent of trip cancellation.
H1: They are associated.

In [2]:
trips["surge_tier"] = np.where(trips.surge_multiplier >= 1.3, "High Surge", "Low/No Surge")
contingency = pd.crosstab(trips.surge_tier, trips.is_cancelled)
chi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)

n = contingency.values.sum()
min_dim = min(contingency.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

cancel_rate_high = trips.loc[trips.surge_tier == "High Surge", "is_cancelled"].mean() * 100
cancel_rate_low = trips.loc[trips.surge_tier == "Low/No Surge", "is_cancelled"].mean() * 100

conclusion = (
    f"Cancellation rate is {cancel_rate_high:.1f}% under high surge vs {cancel_rate_low:.1f}% "
    f"otherwise — {'a statistically significant' if p_chi2 < ALPHA else 'not a statistically significant'} "
    f"association (χ²={chi2:.2f}, p={p_chi2:.4g})."
)
record("Chi-Square Test of Independence", "Does surge pricing associate with cancellation behavior?",
       chi2, p_chi2, cramers_v, "Cramér's V", conclusion)
print(conclusion)

Cancellation rate is 29.7% under high surge vs 17.5% otherwise — a statistically significant association (χ²=839.62, p=1.313e-184).


## 2. Welch's Independent T-Test — Weekend vs. weekday daily demand

H0: Mean daily trip volume is equal on weekends and weekdays.
H1: They differ.

In [3]:
daily_counts = trips.groupby([trips.request_datetime.dt.date, "is_weekend"]).size().reset_index(name="trips")
weekend_daily = daily_counts.loc[daily_counts.is_weekend == 1, "trips"]
weekday_daily = daily_counts.loc[daily_counts.is_weekend == 0, "trips"]

t_stat, p_ttest = stats.ttest_ind(weekend_daily, weekday_daily, equal_var=False)

pooled_sd = np.sqrt((weekend_daily.var(ddof=1) + weekday_daily.var(ddof=1)) / 2)
cohens_d = (weekend_daily.mean() - weekday_daily.mean()) / pooled_sd

conclusion = (
    f"Average daily trips: weekend={weekend_daily.mean():.1f}, weekday={weekday_daily.mean():.1f} "
    f"— {'a statistically significant' if p_ttest < ALPHA else 'not a statistically significant'} "
    f"difference (t={t_stat:.2f}, p={p_ttest:.4g}, Cohen's d={cohens_d:.2f})."
)
record("Welch's Independent T-Test", "Does weekend demand differ significantly from weekday demand?",
       t_stat, p_ttest, cohens_d, "Cohen's d", conclusion)
print(conclusion)

Average daily trips: weekend=148.7, weekday=148.4 — not a statistically significant difference (t=0.17, p=0.8688, Cohen's d=0.02).


## 3. Pearson Correlation — Distance vs. fare

In [4]:
r_dist_fare, p_dist_fare = stats.pearsonr(completed.distance_km, completed.fare_amount)
conclusion = (
    f"Distance and fare are {'strongly' if abs(r_dist_fare) > 0.5 else 'moderately' if abs(r_dist_fare) > 0.3 else 'weakly'} "
    f"correlated (r={r_dist_fare:.3f}, p={p_dist_fare:.4g})."
)
record("Pearson Correlation", "Is trip distance associated with fare?",
       r_dist_fare, p_dist_fare, r_dist_fare, "Pearson r", conclusion)
print(conclusion)

Distance and fare are strongly correlated (r=0.585, p=0).


## 4. Pearson Correlation — Duration vs. fare

In [5]:
r_dur_fare, p_dur_fare = stats.pearsonr(completed.duration_min, completed.fare_amount)
conclusion = (
    f"Duration and fare are {'strongly' if abs(r_dur_fare) > 0.5 else 'moderately' if abs(r_dur_fare) > 0.3 else 'weakly'} "
    f"correlated (r={r_dur_fare:.3f}, p={p_dur_fare:.4g})."
)
record("Pearson Correlation", "Does trip duration influence fare?",
       r_dur_fare, p_dur_fare, r_dur_fare, "Pearson r", conclusion)
print(conclusion)

Duration and fare are strongly correlated (r=0.563, p=0).


## 5. Point-Biserial Correlation — Driver rating vs. cancellation

H0: Driver rating is not associated with whether the trip is cancelled.

In [6]:
r_pb, p_pb = stats.pointbiserialr(trips.is_cancelled, trips.driver_avg_rating)
conclusion = (
    f"Driver rating and cancellation are {'significantly' if p_pb < ALPHA else 'not significantly'} "
    f"associated (r_pb={r_pb:.3f}, p={p_pb:.4g}); lower-rated drivers see more cancellations."
)
record("Point-Biserial Correlation", "Is driver rating associated with cancellation behavior?",
       r_pb, p_pb, r_pb, "Point-biserial r", conclusion)
print(conclusion)

Driver rating and cancellation are significantly associated (r_pb=-0.076, p=4.469e-58); lower-rated drivers see more cancellations.


## 6. Multiple Linear Regression — Fare as a function of trip characteristics

In [7]:
X = completed[["distance_km", "duration_min", "surge_multiplier"]].copy()
X["is_peak_hour"] = completed["is_peak_hour"]
X = sm.add_constant(X)
y = completed["fare_amount"]

ols_model = sm.OLS(y, X).fit()
print(ols_model.summary())

f_stat = ols_model.fvalue
f_pvalue = ols_model.f_pvalue
r_squared = ols_model.rsquared
conclusion = (
    f"Distance, duration, surge multiplier and peak-hour flag jointly explain "
    f"{r_squared*100:.1f}% of fare variance (R²={r_squared:.3f}, F={f_stat:.1f}, p={f_pvalue:.4g})."
)
record("Multiple Linear Regression", "How do distance, duration, surge and peak-hour jointly predict fare?",
       f_stat, f_pvalue, r_squared, "R-squared", conclusion)
print(conclusion)

                            OLS Regression Results                            
Dep. Variable:            fare_amount   R-squared:                       0.390
Model:                            OLS   Adj. R-squared:                  0.390
Method:                 Least Squares   F-statistic:                     5666.
Date:                Sat, 08 Aug 2026   Prob (F-statistic):               0.00
Time:                        17:15:08   Log-Likelihood:            -2.0682e+05
No. Observations:               35507   AIC:                         4.136e+05
Df Residuals:                   35502   BIC:                         4.137e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const              -94.9783      3.773  

## 7. Confidence Interval — Mean fare (95%)

In [8]:
mean_fare = completed.fare_amount.mean()
sem_fare = stats.sem(completed.fare_amount)
ci_low, ci_high = stats.t.interval(0.95, len(completed) - 1, loc=mean_fare, scale=sem_fare)
conclusion = f"95% CI for mean fare: [Rs {ci_low:.2f}, Rs {ci_high:.2f}] (point estimate Rs {mean_fare:.2f})."
record("95% Confidence Interval", "What is the plausible range for true mean fare?",
       mean_fare, np.nan, ci_high - ci_low, "CI width (Rs)", conclusion)
print(conclusion)

95% CI for mean fare: [Rs 142.75, Rs 144.93] (point estimate Rs 143.84).


## Save results

In [9]:
results_df = pd.DataFrame(results)
results_df.to_csv(PROCESSED_DIR / "statistical_test_results.csv", index=False)
print(f"\nSaved {len(results_df)} test results -> data/processed/statistical_test_results.csv")
results_df[["test_name", "statistic", "p_value", "significant_at_0.05"]]


Saved 7 test results -> data/processed/statistical_test_results.csv


,test_name,statistic,p_value,significant_at_0.05
0,Chi-Square Test of Independence,839.6197,0.000000,True
1,Welch's Independent T-Test,0.1656,0.868752,False
2,Pearson Correlation,0.5848,0.000000,True
3,Pearson Correlation,0.5626,0.000000,True
4,Point-Biserial Correlation,-0.0756,0.000000,True
5,Multiple Linear Regression,5665.9676,0.000000,True
6,95% Confidence Interval,143.8409,NaN,False
